# Convokit Regression Analysis
Runs all six regression models from Chapter 4 of the methodology on the combined
Convokit lexical CSV.

**Key differences from the ArcticShift version:**
- `controversiality` and `edited` are not available in Convokit data; the three
  utterance-level models (WLS, FE, mixed-effects) will use the reduced control set
  `[post_depth, score, num_direct_replies]` automatically.
- `year_month` and `log_freq_month` are not pre-computed in the CSV and are derived
  below from `timestamp` and `num_utterances_by_speaker_month` respectively.

In [1]:
import warnings
import numpy as np
import pandas as pd

from regressions import (
    run_baseline_ols,
    run_first_diff_ols,
    run_ar_ols,
    run_cross_user_wls,
    run_fixed_effects_panel,
    run_mixed_effects,
    LEXICAL_METRICS,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CSV_PATH = "/scratch/network/nv9344/Thesis/Thesis-Data/Convokit/lexical_master.csv"

METRICS  = LEXICAL_METRICS   # all six, or pass a subset to any function below
ALPHA    = 0.05              # significance level
APPLY_BH = True              # Benjamini-Hochberg FDR correction

In [2]:
# Load and preprocess Convokit data
# raw_text is excluded via usecols — not needed for any regression.
# controversiality and edited are absent from Convokit; utterance-level models
# (WLS, FE, mixed-effects) handle this automatically via avail_controls filtering.
COLS_TO_LOAD = [
    "utterance_id", "speaker_id", "subreddit",
    "timestamp",
    "num_utterances_by_speaker", "num_utterances_by_speaker_month",
    "post_depth", "score", "num_direct_replies",
    "mtld_score", "mattr_score", "yules_k", "zipf_score", "aoa_score", "nawl_ratio",
]

df = pd.read_csv(CSV_PATH, usecols=COLS_TO_LOAD, low_memory=False)

# Derive year_month from timestamp (stored as "YYYY-MM-DD HH:MM:SS" strings)
df["year_month"] = pd.to_datetime(df["timestamp"]).dt.to_period("M").astype(str)

# Compute log monthly posting frequency (F_ut) — required by WLS, FE, and mixed-effects.
# log1p avoids -inf for any speaker-months with zero recorded utterances.
df["log_freq_month"] = np.log1p(df["num_utterances_by_speaker_month"])

print(f"Loaded {len(df):,} utterances across {df['subreddit'].nunique()} subreddit(s).")
print(df[["subreddit", "year_month"]].groupby("subreddit")["year_month"].nunique()
        .rename("n_months").to_frame().T)

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/network/nv9344/Thesis/Thesis-Data/Convokit/lexical_master.csv'

# Baseline OLS Regression
Model: `y_t = β₀ + β₁·t + ε_t` — one regression per (subreddit × metric), fitted on
monthly-aggregated data with Newey-West HAC standard errors. β₁ is the estimated
change in the metric per calendar month.

In [ ]:
ols_results = run_baseline_ols(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

# Summary counts
print("=== Baseline OLS — conclusion counts ===")
print(ols_results["conclusion"].value_counts().to_string())
print()

# Significant results
sig = ols_results[ols_results["significant"] == True].copy()
print(f"Significant (subreddit × metric) pairs: {len(sig)} / {len(ols_results)}")
if len(sig):
    print(sig[["subreddit", "metric", "beta_1", "se_beta_1", "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

ols_results

# First-Differenced OLS Regression
Model: `Δy_t = β₀ + ε_t` — the drift model fitted on first-differenced monthly series
with HAC standard errors. A significant β₀ (drift) with the same sign as β₁ from the
baseline levels regression provides convergent evidence of a genuine trend.

In [ ]:
fd_results = run_first_diff_ols(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== First-Differenced OLS — conclusion counts ===")
print(fd_results["conclusion"].value_counts().to_string())
print()

sig_fd = fd_results[fd_results["significant"] == True].copy()
print(f"Significant pairs: {len(sig_fd)} / {len(fd_results)}")
if len(sig_fd):
    print(sig_fd[["subreddit", "metric", "drift", "se_drift", "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

# Convergence check: compare signs with baseline OLS
merged = ols_results[["subreddit", "metric", "beta_1", "significant"]].rename(
    columns={"beta_1": "ols_beta1", "significant": "ols_sig"}
).merge(
    fd_results[["subreddit", "metric", "drift", "significant"]].rename(
        columns={"drift": "fd_drift", "significant": "fd_sig"}
    ),
    on=["subreddit", "metric"],
)
merged["signs_agree"] = (
    merged["ols_beta1"].apply(lambda x: 1 if x > 0 else -1) ==
    merged["fd_drift"].apply(lambda x: 1 if x > 0 else -1)
)
print()
print("=== OLS ↔ First-Diff sign convergence ===")
print(merged[["subreddit", "metric", "ols_beta1", "ols_sig", "fd_drift", "fd_sig", "signs_agree"]]
      .sort_values(["metric", "subreddit"])
      .to_string(index=False))

fd_results

# AR OLS Regression
Model: `y_t = β₀ + β₁·t + φ·y_{t-1} + ε_t` — adds an AR(1) term to the baseline to
absorb autocorrelation, with HAC standard errors. β₁ here captures the trend net of
persistence; φ captures the autoregressive pull of the previous month.

In [ ]:
ar_results = run_ar_ols(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== AR OLS — conclusion counts ===")
print(ar_results["conclusion"].value_counts().to_string())
print()

sig_ar = ar_results[ar_results["significant"] == True].copy()
print(f"Significant pairs: {len(sig_ar)} / {len(ar_results)}")
if len(sig_ar):
    print(sig_ar[["subreddit", "metric", "beta_1", "se_beta_1", "phi",
                  "p_value_beta1", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

# Three-model sign convergence: OLS, FD, AR all agree → strong evidence
convergence = merged.merge(
    ar_results[["subreddit", "metric", "beta_1", "significant"]].rename(
        columns={"beta_1": "ar_beta1", "significant": "ar_sig"}
    ),
    on=["subreddit", "metric"],
)
convergence["all_agree"] = (
    convergence["signs_agree"] &
    (convergence["ols_beta1"].apply(lambda x: 1 if x > 0 else -1) ==
     convergence["ar_beta1"].apply(lambda x: 1 if x > 0 else -1))
)
print()
print("=== Three-model sign convergence (OLS, FD, AR) ===")
print(convergence[["subreddit", "metric", "ols_sig", "fd_sig", "ar_sig", "all_agree"]]
      .sort_values(["all_agree", "metric", "subreddit"], ascending=[False, True, True])
      .to_string(index=False))

ar_results

# Cross-User WLS Regression
Model: `ȳ_u = β₀ + β₁·F̄_u + β₂·X̄_u + ε_u` — user-level means, one regression per
(subreddit × metric), weighted by total post count n_u. β₁ captures the relationship
between posting frequency and lexical quality across users.

Controls used: `post_depth`, `score`, `num_direct_replies`
(`edited` and `controversiality` are absent from Convokit and are silently dropped.)

In [ ]:
wls_results = run_cross_user_wls(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== Cross-User WLS — conclusion counts ===")
print(wls_results["conclusion"].value_counts().to_string())
print()

sig_wls = wls_results[wls_results["significant"] == True].copy()
print(f"Significant pairs: {len(sig_wls)} / {len(wls_results)}")
if len(sig_wls):
    print(sig_wls[["subreddit", "metric", "n_users", "beta_1", "se_beta_1",
                   "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

wls_results

# Fixed Effects Panel Regression
Model: `y_ust = β₁·F_ut + β₂·X_ust + α_u + γ_t + δ_s + ε_ust` — user, time, and
subreddit fixed effects across all communities jointly. β₁ is the within-user effect
of log monthly posting frequency on lexical quality.

Controls used: `post_depth`, `score`, `num_direct_replies`
(`edited` and `controversiality` are absent from Convokit and are silently dropped.)

In [ ]:
fe_results = run_fixed_effects_panel(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== Fixed Effects Panel — conclusion counts ===")
print(fe_results["conclusion"].value_counts().to_string())
print()

sig_fe = fe_results[fe_results["significant"] == True].copy()
print(f"Significant metrics: {len(sig_fe)} / {len(fe_results)}")
if len(sig_fe):
    print(sig_fe[["metric", "n_obs", "n_users", "n_periods",
                  "beta_1", "se_beta_1", "p_value", "p_value_bh", "conclusion"]]
          .to_string(index=False))

fe_results

# Cross-Subreddit Mixed-Effects Model
Model: `y_ust = Σ_s δ_s·1[subreddit=s] + β·X_ust + α_u + ε_ust` — subreddit fixed
effects with user random intercepts (REML). δ_s is the conditional mean difference in
lexical quality for subreddit s relative to the reference (first alphabetically).

Controls used: `post_depth`, `score`, `num_direct_replies`
(`edited` and `controversiality` are absent from Convokit and are silently dropped.)

In [ ]:
mixed_results = run_mixed_effects(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== Mixed Effects — conclusion counts ===")
print(mixed_results["conclusion"].value_counts().to_string())
print()

ref = mixed_results["reference_subreddit"].iloc[0] if len(mixed_results) else "N/A"
print(f"Reference subreddit: {ref}")
print()

sig_mixed = mixed_results[mixed_results["significant"] == True].copy()
print(f"Significant (metric × subreddit) pairs: {len(sig_mixed)} / {len(mixed_results)}")
if len(sig_mixed):
    print(sig_mixed[["metric", "subreddit", "reference_subreddit",
                     "delta", "se_delta", "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

mixed_results